# mT5 LoRA fine-tuning — MWE paraphrasing (PARSEME 2.0 Subtask 2)

Multilingual seq2seq SFT with PEFT/LoRA on `google/mt5-large`.

- **Input format:** `paraphrase <LANG>: {sentence}`
- **Target:** `{paraphrase}`
- **Save location:** `MyDrive/mt5_mwe/checkpoints/`
- **Tier assumed:** Colab Pro (L4 24 GB or A100 40 GB)

Runtime → *Change runtime type* → GPU (L4 or A100) before running.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.41" "peft>=0.11" "accelerate>=0.30" \
                "datasets>=2.18" "sentencepiece" "tqdm" "bert-score"
# Colab ships an old torchao (0.10.0) that current PEFT rejects on import.
# We don't use torchao — uninstall so PEFT's dispatcher silently skips that path.
!pip -q uninstall -y torchao

## 2. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive/mt5_mwe'
DATA_DIR    = os.path.join(DRIVE_ROOT, 'data')
CKPT_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
ADAPTER_DIR = os.path.join(DRIVE_ROOT, 'adapter_final')
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
print('Data dir   :', DATA_DIR)
print('Checkpoints:', CKPT_DIR)

## 3. (Optional) One-off converter for `ai_studio_code.txt`

Run this cell ONCE if you've uploaded the original FR file to `MyDrive/mt5_mwe/data/ai_studio_code.txt`.
It writes `data/fr.json` in the canonical schema and skips rows where `text == paraphrase`.

In [ ]:
import json
src = os.path.join(DATA_DIR, 'ai_studio_code.txt')
dst = os.path.join(DATA_DIR, 'fr.json')

if os.path.exists(src):
    with open(src, encoding='utf-8') as f:
        raw = json.load(f)
    out = []
    for r in raw:
        s, p = r.get('text', '').strip(), r.get('paraphrase', '').strip()
        if s and p and s != p:
            out.append({'language': 'FR', 'sentence': s, 'paraphrase': p})
    with open(dst, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f'Wrote {len(out)} pairs \u2192 {dst}')
else:
    print(f'Source not found at {src}; skipping (already converted, or upload fr.json directly).')

## 4. Hyperparameters

Tune these in one place. Defaults target Colab Pro on L4 24 GB.

In [ ]:
MODEL_NAME       = 'google/mt5-large'   # try 'google/mt5-base' if OOM, 'google/mt5-xl' on A100 40GB
MAX_INPUT_LEN    = 192
MAX_TARGET_LEN   = 192

# LoRA
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGETS     = ['q', 'v']           # q/v projections of attention

# Optimization
EPOCHS           = 5
PER_DEVICE_BS    = 8
GRAD_ACCUM       = 4                    # effective batch 32
LR               = 3e-4                 # standard for LoRA; ~10x higher than full-FT
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
LABEL_SMOOTHING  = 0.1

# Eval/save
VAL_FRACTION     = 0.10
SEED             = 42
LOG_STEPS        = 10
SAVE_TOTAL_LIMIT = 2

## 5. Load and normalize the data

Globs every `*.json` under `DATA_DIR` and concatenates. Each file must be a list of
`{"language": "XX", "sentence": "...", "paraphrase": "..."}`.

In [ ]:
import json, glob, random
from collections import Counter

def load_all_pairs(data_dir: str):
    pairs = []
    files = sorted(glob.glob(os.path.join(data_dir, '*.json')))
    for fp in files:
        with open(fp, encoding='utf-8') as f:
            try:
                rows = json.load(f)
            except json.JSONDecodeError as e:
                print(f'  ! skipping {fp}: {e}')
                continue
        if not isinstance(rows, list):
            print(f'  ! skipping {fp}: not a list')
            continue
        n_before = len(pairs)
        for r in rows:
            lang = (r.get('language') or '').upper().strip()
            s    = (r.get('sentence') or '').strip()
            p    = (r.get('paraphrase') or '').strip()
            if lang and s and p and s != p:
                pairs.append({'language': lang, 'sentence': s, 'paraphrase': p})
        print(f'  {os.path.basename(fp):30s} \u2192 {len(pairs)-n_before} valid pairs')
    return pairs

pairs = load_all_pairs(DATA_DIR)
print(f'\nTotal valid pairs: {len(pairs)}')
print('By language:', dict(Counter(r["language"] for r in pairs)))

assert len(pairs) > 0, f'No data found in {DATA_DIR}. Upload JSON files first.'

## 6. Train/val split

In [ ]:
from datasets import Dataset

random.seed(SEED)
random.shuffle(pairs)

def to_io(r):
    return {
        'input_text' : f"paraphrase <{r['language']}>: {r['sentence']}",
        'target_text': r['paraphrase'],
    }

io_pairs = [to_io(r) for r in pairs]
n_val    = max(1, int(len(io_pairs) * VAL_FRACTION))
val_set  = io_pairs[:n_val]
train_set= io_pairs[n_val:]
print(f'train: {len(train_set)} | val: {len(val_set)}')

train_ds = Dataset.from_list(train_set)
val_ds   = Dataset.from_list(val_set)

## 7. Tokenizer + tokenization

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print('Vocab size:', tokenizer.vocab_size)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(  tokenize_batch, batched=True, remove_columns=val_ds.column_names)
print(train_tok)

## 8. Load mT5 + apply LoRA

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype    = torch.bfloat16 if use_bf16 else torch.float16
print('CUDA  :', torch.cuda.is_available(), '| bf16:', use_bf16)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))

base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)

lora_cfg = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = LORA_TARGETS,
    bias           = 'none',
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

## 9. Trainer setup

`Seq2SeqTrainer` already shows a tqdm progress bar per epoch.

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding='longest')

args = Seq2SeqTrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BS,
    per_device_eval_batch_size  = PER_DEVICE_BS,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    label_smoothing_factor      = LABEL_SMOOTHING,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    bf16                        = use_bf16,
    fp16                        = (not use_bf16) and torch.cuda.is_available(),
    gradient_checkpointing      = True,
    logging_steps               = LOG_STEPS,
    predict_with_generate       = False,    # speed up eval; we'll generate manually after
    report_to                   = 'none',
    seed                        = SEED,
    disable_tqdm                = False,
)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = args,
    train_dataset = train_tok,
    eval_dataset  = val_tok,
    processing_class = tokenizer,
    data_collator = collator,
)

## 10. Train

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 11. Save the LoRA adapter, tokenizer, and run config

Adapter weights only — you load them on top of stock `google/mt5-large` later via `PeftModel.from_pretrained`.

In [ ]:
import json, datetime

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

run_meta = {
    'model_name'      : MODEL_NAME,
    'lora'            : {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
                          'targets': LORA_TARGETS},
    'epochs'          : EPOCHS,
    'effective_bs'    : PER_DEVICE_BS * GRAD_ACCUM,
    'lr'              : LR,
    'max_input_len'   : MAX_INPUT_LEN,
    'max_target_len'  : MAX_TARGET_LEN,
    'n_train'         : len(train_tok),
    'n_val'           : len(val_tok),
    'languages'       : sorted(set(r['language'] for r in pairs)),
    'final_metrics'   : train_result.metrics,
    'saved_at'        : datetime.datetime.utcnow().isoformat() + 'Z',
}
with open(os.path.join(ADAPTER_DIR, 'run_meta.json'), 'w') as f:
    json.dump(run_meta, f, indent=2, default=str)

print('Saved adapter \u2192', ADAPTER_DIR)
print('Files       :', os.listdir(ADAPTER_DIR))

## 12. Inference smoke test

Generate paraphrases for the first few validation examples to sanity-check the trained adapter.

In [ ]:
from tqdm.auto import tqdm

model.eval()
device = next(model.parameters()).device
n_show = min(8, len(val_set))

print(f'\nGenerating {n_show} samples on {device}\n' + '\u2500' * 60)
for ex in tqdm(val_set[:n_show], desc='generating'):
    enc = tokenizer(ex['input_text'], return_tensors='pt',
                    max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens = MAX_TARGET_LEN,
            num_beams      = 4,
            early_stopping = True,
            no_repeat_ngram_size = 3,
        )
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"INPUT : {ex['input_text']}")
    print(f"GOLD  : {ex['target_text']}")
    print(f"PRED  : {pred}")
    print('\u2500' * 60)

## 13. (Optional) Reload later from Drive

After a runtime restart, you don't need to retrain — just reload base + adapter:

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-large', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, '/content/drive/MyDrive/mt5_mwe/adapter_final')
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/mt5_mwe/adapter_final')
```